# Building the Core: A Deep Learning Model for Dimension Estimation

In [ ]:
# Here the code will construct, train, and evaluate a machine learning model that predicts a product's dimensions (length, width, height) from its image. 

# We will use a powerful technique called transfer learning, which involves adapting a pre-trained Convolutional Neural Network (CNN) for our specific task.
# Here we will use the MobileNetV2 architecture, known for its efficiency and strong performance, implemented with the TensorFlow and Keras libraries.

# MobileNetV2
# - A lightweight CNN architecture developed by Google specifically designed for mobile and embedded vision applications
# - It uses depthwise separable convolutions to reduce the number of parameters and computations.

# Transfer Learning
# - NLP models (and others) are too big and complex to build from scratch and re-train every time.
# - Thus better is start from pre-trained models and fine-tune these models for your own use cases. This is called as Transfer Learning
# - Approaches
# 	- Continue training a pre-trained model (fine-tuning)
# 	- Add new trainable layers to the top of a frozen model
# 	- Retrain from scratch
# 	- Use it as-is

In [ ]:
# Prerequistie Installs
# pip install tensorflow pandas numpy scikit-learn pillow

# You should also have the preprocessed data from Step 1, (01_imagemassgeneration.ipynb) specifically:
    # A CSV file (e.g., input_image2mass_images/image2mass_ground_truth.csv) with image identifiers and their corresponding dimensions.
    # A directory (e.g., preprocessed_image2mass_images) containing the normalized image data saved as .npy files.

In [ ]:
import os
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from PIL import Image

In [ ]:
# --- 2.1. Configuration, Data (Images) Loading, Convert to NumPy Arrays, Data Splitting ---

In [ ]:
# Configuration
DATA_DIR = 'preprocessed_raw_images'
CSV_FILE = 'input_raw_images/raw_product_dataset.csv' # The CSV from Step 1 (01_imagemassgeneration.ipynb) 
IMAGE_DIMS = (224, 224, 3) # Must match the dimensions from preprocessing
TEST_SPLIT_SIZE = 0.2 # 20% of the data is reserved for validation, ensuring the model generalizes well.
RANDOM_STATE = 42
LEARNING_RATE = 0.001 
EPOCHS = 25  # The model trains 25 times over the entire dataset.
BATCH_SIZE = 32 # The dataset will split into mini-batches of 32 samples for training efficiency.

print("Loading dataset...")

# Load the ground truth data
try:
    df = pd.read_csv(CSV_FILE)
    print(f"Loaded {len(df)} entries from {CSV_FILE}")
    # print(df.head())
except FileNotFoundError:
    print(f"Error: The file '{CSV_FILE}' was not found. Please run Step 1 first.")
    exit()

# Prepare file paths and labels
# NOTE: This part assumes your preprocessed files are named `normalized_product_X.npy`
# and correspond to the rows in your CSV. Adjust if your naming is different.
image_files = []
dimensions = []

for index, row in df.iterrows():
    # Construct the expected numpy filename from the preprocessing step
    # npy_filename = os.path.join(DATA_DIR, f"normalized_product_{index + 1}.npy")

    filename = row['imagepath']
    filename = os.path.splitext(filename)[0]
    npy_filename = os.path.join(DATA_DIR, f"normalized_" + filename + ".npy")
    # splitext(filename)[0]

    if os.path.exists(npy_filename):
        image_files.append(npy_filename)
        # We want to predict length, width, and height
        # dimensions.append(row[['length_cm', 'width_cm', 'height_cm']].values)
        dimensions.append(row[['product_length', 'product_width', 'product_height']].values)
    else:
        print(f"Warning: Could not find {npy_filename}. Skipping this entry.")

# Convert to NumPy arrays. Converting data to NumPy arrays in Python is often essential because 
# NumPy is the foundational library for numerical computing—especially in data science, machine learning, and scientific computing.

# When working with images (like in your Image2Mass workflow), converting to NumPy arrays allows:
    # Pixel-level manipulation
    # Feeding data into ML models
    # Efficient resizing, filtering, and normalization

# Example: img = Image.open("sample.jpg")
# img_array = np.array(img)  # Converts to shape (H, W, C)
# In image processing and computer vision, (H, W, C) refers to the shape of an image array:
    # H = Height (number of pixels vertically)
    # W = Width (number of pixels horizontally)
    # C = Channels (number of color components per pixel)

X = np.array([np.load(file) for file in image_files])
y = np.array(dimensions, dtype=np.float32)

print(f"Dataset loaded successfully. Found {len(X)} matching images and labels.")

# Split the data into training and testing sets
# Splitting data into training and testing sets is a foundational practice in machine learning—and it’s absolutely critical for building models that generalize well to unseen data
# Here’s why:
    # The training set is used to teach the model—adjusting weights, learning patterns, and minimizing error.
    # The testing set is used to evaluate how well the model performs on new, unseen data.
    # This split simulates real-world deployment, where the model will encounter data it hasn’t seen before.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SPLIT_SIZE, random_state=RANDOM_STATE
)

print(f"Data split into {len(X_train)} training samples and {len(X_test)} testing samples.")
print(f"Data split into {len(y_train)} training samples and {len(y_test)} testing samples.")


In [ ]:
# --- 2.2. Building the Model with Transfer Learning ---

In [ ]:
print("\nBuilding the model...")

# Load the MobileNetV2 base model, pre-trained on ImageNet
# include_top=False means we don't include the final classification layer
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_tensor=Input(shape=IMAGE_DIMS)
)

# Freeze the base model layers to prevent them from being updated during training
base_model.trainable = False

# Add custom layers on top of the base model
# Create the custom regression "head" to place on top of the base model
# This part of the model will be trained.
x = base_model.output
x = GlobalAveragePooling2D()(x) # Averages the spatial features
x = Dense(128, activation='relu')(x)   # A dense layer for learning complex relationships
x = Dense(64, activation='relu')(x)    # Another dense layer
# The final output layer has 3 neurons (for length, width, height) and a linear activation
# because we are predicting continuous values (regression).
predictions = Dense(3, activation='linear')(x)

# Create the final model (Combine the base model and our custom head into the final model)
product_size_prediction_model = Model(inputs=base_model.input, outputs=predictions)

# Display the model's architecture
product_size_prediction_model.summary()

In [ ]:
# --- 2.3. Compiling the Model ---

In [ ]:
print("\nCompiling the model...")
# For regression, 'mean_squared_error' is a common loss function.
# 'mean_absolute_error' gives us a more interpretable metric of how far off our predictions are on average.
product_size_prediction_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='mean_squared_error',
    metrics=['mean_absolute_error']
)

In [ ]:
# --- 2.4. Training the Model ---

In [ ]:
print("\nTraining the model...")
history = product_size_prediction_model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1
)

# ---------------------------------------------------------------------------------------
# This statement (model.fit) is used to 'TRAIN A DEEP LEARNING MODEL' in TensorFlow/Keras.

# Breakdown of Parameters:
# model.fit(X, y, epochs=25, batch_size=32, validation_split=0.2)

    # working_features_scaled → The input features (X) after scaling.
    # predict_features → The target labels (y) the model is trying to predict.
    # epochs=25 → The model trains 25 times over the entire dataset.
    # batch_size=32 → The dataset is split into mini-batches of 32 samples for training efficiency.
    # validation_split=0.2 → 20% of the data is reserved for validation, ensuring the model generalizes well.

# What Happens During Execution?
# The model iterates 50 times over the dataset to learn patterns.
# It processes 32 samples at a time, updating weights after each batch.
# 20% of the training data is set aside to evaluate performance after each epoch.
# ---------------------------------------------------------------------------------------

In [ ]:
# --- 2.5. Evaluating the Model ---

In [ ]:
print("\nEvaluating model performance...")
loss, mae = product_size_prediction_model.evaluate(X_test, y_test, verbose=0)
print(f"Test Set Mean Absolute Error: {mae:.2f} cm")
print("This means, on average, the model's dimension predictions are off by about {:.2f} cm.".format(mae))

# ---------------------------------------------------------------------------------------
# This statement (model.evaluate) 'EVALUATES THE TRAINED MODEL'S PERFORMANCE' on a given dataset.
# Breakdown:
    # model.evaluate(X_test, y_test) → Runs the trained model on test data and computes the loss & metrics.
    # X_test → The input testing samples
    # y_test → The actual testing samples

# Returns:
    # loss → The error of the model based on the loss function (e.g., binary cross-entropy).
    # accuracy → The accuracy of the model, as defined in model.compile().

# What Happens?
    # The model processes the given dataset without updating weights.
    # It computes the loss based on the selected loss function.
    # It calculates accuracy as a performance metric.
    # The final loss and accuracy values are stored and can be printed.
# ---------------------------------------------------------------------------------------

In [ ]:
# --- 2.6. Saving the Model for Future Use ---
product_size_prediction_model.save("02_image_dimension_estimation_model.h5")
print("\nModel saved to '02_image_dimension_estimation_model.h5'.")

In [ ]:
# --- 7. Function Prediction on a New Image ---

In [ ]:
def predict_product_dimensions(image_path, model_to_use):
    """
    Takes an image path, preprocesses it, and predicts its dimensions.
    """
    try:
        # Preprocess the new image in the same way as the training data
        img = Image.open(image_path).convert('RGB')
        resized_img = img.resize((IMAGE_DIMS[0], IMAGE_DIMS[1]))
        normalized_array = np.array(resized_img) / 255.0

        # The model expects a batch of images, so we add an extra dimension
        input_data = np.expand_dims(normalized_array, axis=0)

        # Make the prediction
        predicted_dims = model_to_use.predict(input_data)[0]
        return {
            "length_cm": predicted_dims[0],
            "width_cm": predicted_dims[1],
            "height_cm": predicted_dims[2]
        }
    except FileNotFoundError:
        return f"Error: Image not found at {image_path}"
    except Exception as e:
        return f"An error occurred: {e}"

In [ ]:
# 2.8 Example usage

In [ ]:
image_path = "image_to_predict/81b93e5d.jpg"
predicted_dimensions = predict_product_dimensions(image_path, product_size_prediction_model)
print(predicted_dimensions) 

In [ ]:

# Create a dummy image for prediction if you don't have one.
# Replace 'path/to/your/random/product_image.jpg' with a real image path.
# For this example, let's just use one of our test images.
if len(X_test) > 0:
    # Get the filename corresponding to the first test image
    first_test_index = np.where((X == X_test[0]).all(axis=(1,2,3)))[0][0]
    # original_image_path = os.path.join(
    #     'input_image2mass_images', # Original image folder
    #     f"product_{df.index[first_test_index] + 1}.jpg"
    # )

    original_image_path = os.path.join(
    'input_raw_images', # Original image folder
    f"81a0b666.jpg")

    print(original_image_path)
    
    print("\n--- Example Prediction ---")
    predicted_dimensions = predict_product_dimensions(original_image_path, product_size_prediction_model)

    if isinstance(predicted_dimensions, dict):
        print(f"Image: {original_image_path}")
        print(f"Predicted Dimensions -> Length: {predicted_dimensions['length_cm']:.2f} cm, "
              f"Width: {predicted_dimensions['width_cm']:.2f} cm, "
              f"Height: {predicted_dimensions['height_cm']:.2f} cm")

        # Compare with actual dimensions (y_test)
        actual_dims = y_test[0]
        print(f"Actual Dimensions    -> Length: {actual_dims[0]:.2f} cm, "
              f"Width: {actual_dims[1]:.2f} cm, "
              f"Height: {actual_dims[2]:.2f} cm")